# Preprocesamiento y Modelado — Telco Customer Churn

## Contenido
1. Limpieza de datos
2. División train/test
3. Función reutilizable con Pipeline
4. Entrenamiento de modelos
5. Ranking comparativo
6. Evaluación de modelos
7. Guardado del modelo final y archivos para LIME

## 1. Limpieza de datos

En el EDA identificamos que `TotalCharges` viene como string en el dataset original de Kaggle y los clientes con `tenure = 0` tienen `TotalCharges` vacío. Se corrige aquí antes de construir el Pipeline.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import time
import json
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

pd.options.display.float_format = '{:,.4f}'.format

In [10]:
df = pd.read_csv('../data/telco-churn.csv')

# Convertir TotalCharges a numérico (viene como string en el dataset original)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Imputar NaN en TotalCharges con 0 (clientes con tenure=0 no tienen cargos)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convertir variable objetivo a binario
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Eliminar columna de identificación
df = df.drop(columns=['customerID'], errors='ignore')

print(f'Filas después de limpieza: {df.shape[0]}')
print(f'NaN en TotalCharges: {df["TotalCharges"].isnull().sum()}')
print(f'Distribución Churn: {df["Churn"].value_counts().to_dict()}')

Filas después de limpieza: 7043
NaN en TotalCharges: 0
Distribución Churn: {0: 5174, 1: 1869}


> **Análisis:** Describir el estado del dataset tras la limpieza.

## 2. División train/test

Se dividen los datos en 80% entrenamiento y 20% prueba de forma **estratificada** para preservar la distribución del target en ambos conjuntos.

In [11]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Churn en train: {y_train.mean():.3f} | Churn en test: {y_test.mean():.3f}')

Train: (5634, 19), Test: (1409, 19)
Churn en train: 0.265 | Churn en test: 0.265


> **Análisis:** Verificar que la proporción de churn se mantiene similar en train y test (estratificación correcta).

## 3. Función reutilizable con Pipeline

Se define un `ColumnTransformer` que aplica:
- `OneHotEncoder` a las variables categóricas — las variables categóricas de Telco no tienen orden natural (`PaymentMethod`, `Contract`, `InternetService`, etc.), por lo que OrdinalEncoder introduciría jerarquías numéricas falsas. OHE genera columnas binarias (0/1) limpias para cada categoría.
- `StandardScaler` a las variables numéricas

Todo dentro de un `Pipeline` para evitar data leakage: las transformaciones se aprenden **solo** sobre `X_train` y se aplican a `X_test`.

In [12]:
# Identificar tipos de columnas
categoricas = X.select_dtypes(include='object').columns.tolist()
numericas   = X.select_dtypes(include='number').columns.tolist()

print(f'Numéricas   ({len(numericas)}): {numericas}')
print(f'Categóricas ({len(categoricas)}): {categoricas}')

Numéricas   (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [13]:
def train_pipeline(X_train, y_train, model, param_grid, cv=5):
    """Construye un Pipeline con preprocesamiento y ajusta hiperparámetros con GridSearchCV."""
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categoricas),
        ('num', StandardScaler(), numericas)
    ])

    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf',  model)
    ])

    cv_estratificado = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv_estratificado,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)
    return grid

> **Análisis:** Explicar por qué se usa Pipeline en lugar de transformar el dataset directamente, y qué rol juega `StratifiedKFold` en la validación cruzada.

## 4. Entrenamiento de modelos

Se entrenan cuatro clasificadores con los hiperparámetros adicionales recomendados en el enunciado. Se seleccionaron los primeros 4 de cada modelo para mantener el grid manejable: **16 combinaciones × 5 folds = 80 fits por modelo**, 320 fits en total.

El prefijo `clf__` en cada hiperparámetro es la sintaxis de `GridSearchCV` para indicar a qué paso del Pipeline pertenece el parámetro — en este caso al paso `clf` (el clasificador).

### Justificación de grids

**Random Forest** — `min_samples_split`, `min_samples_leaf`, `max_features`, `bootstrap`
- `min_samples_split [2, 10]`: 2 es el default; 10 evita splits con muy pocos datos en un dataset de 7k filas
- `min_samples_leaf [1, 4]`: 1 es default; 4 añade regularización sin ser muy restrictivo
- `max_features ['sqrt', 'log2']`: ambos son estándar para clasificación; sqrt suele ganar pero log2 reduce correlación entre árboles
- `bootstrap [True, False]`: True es bagging clásico; False usa todo el dataset en cada árbol

**XGBoost** — `subsample`, `colsample_bytree`, `gamma`, `min_child_weight`
- `subsample [0.8, 1.0]`: 0.8 añade estocasticidad que reduce overfitting; 1.0 es default
- `colsample_bytree [0.8, 1.0]`: fracción de features por árbol; 0.8 reduce correlación entre árboles
- `gamma [0, 0.1]`: 0 es default sin penalización; 0.1 exige ganancia mínima para hacer un split
- `min_child_weight [1, 5]`: 1 es default; 5 hace el modelo más conservador con nodos hoja

**CatBoost** — `l2_leaf_reg`, `bagging_temperature`, `border_count`, `random_strength`
- `l2_leaf_reg [1, 5]`: 1 es default; 5 añade regularización L2 para reducir overfitting
- `bagging_temperature [0, 1]`: 0 desactiva el bagging bayesiano; 1 es default con mayor aleatoriedad
- `border_count [32, 128]`: número de splits candidatos por feature; 32 es más rápido, 128 más preciso
- `random_strength [0.5, 1]`: aleatoriedad al seleccionar splits; valores bajos producen modelos más estables

**LightGBM** — `min_child_samples`, `min_child_weight`, `subsample`, `colsample_bytree`
- `min_child_samples [20, 50]`: 20 es default; 50 exige más datos por hoja, reduce overfitting
- `min_child_weight [0.001, 0.01]`: peso mínimo en una hoja; valores más altos hacen el modelo más conservador
- `subsample [0.8, 1.0]`: fracción de filas por árbol; 0.8 añade estocasticidad
- `colsample_bytree [0.8, 1.0]`: fracción de features por árbol

In [ ]:
modelos = {
    'RandomForest': (
        RandomForestClassifier(random_state=42),
        {
            'clf__min_samples_split': [2, 10],
            'clf__min_samples_leaf':  [1, 4],
            'clf__max_features':      ['sqrt', 'log2'],
            'clf__bootstrap':         [True, False]
        }
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
        {
            'clf__subsample':        [0.8, 1.0],
            'clf__colsample_bytree': [0.8, 1.0],
            'clf__gamma':            [0, 0.1],
            'clf__min_child_weight': [1, 5]
        }
    ),
    'CatBoost': (
        CatBoostClassifier(random_seed=42, verbose=0),
        {
            'clf__l2_leaf_reg':        [1, 5],
            'clf__bagging_temperature': [0, 1],
            'clf__border_count':        [32, 128],
            'clf__random_strength':     [0.5, 1]
        }
    ),
    'LightGBM': (
        LGBMClassifier(random_state=42, verbose=-1),
        {
            'clf__min_child_samples': [20, 50],
            'clf__min_child_weight':  [0.001, 0.01],
            'clf__subsample':         [0.8, 1.0],
            'clf__colsample_bytree':  [0.8, 1.0]
        }
    )
}

In [ ]:
grids      = {}
resultados = {}
tiempo_total_inicio = time.time()

for nombre, (modelo, params) in tqdm(modelos.items(), desc='Modelos', unit='modelo'):

    n_combinaciones = 1
    for v in params.values():
        n_combinaciones *= len(v)
    print(f'\n🔄 {nombre} — {n_combinaciones} combinaciones × 5 folds = {n_combinaciones * 5} fits')

    t_inicio = time.time()
    grid = train_pipeline(X_train, y_train, modelo, params)
    elapsed = time.time() - t_inicio

    y_pred  = grid.predict(X_test)
    y_proba = grid.predict_proba(X_test)[:, 1]

    auc       = roc_auc_score(y_test, y_proba)
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)

    best_params_clean = {
        k.replace('clf__', ''): v
        for k, v in grid.best_params_.items()
    }

    resultados[nombre] = {
        'AUC':            round(auc,       4),
        'Accuracy':       round(acc,       4),
        'Precision':      round(precision, 4),
        'Recall':         round(recall,    4),
        'F1':             round(f1,        4),
        'Tiempo (min)':   round(elapsed / 60, 2),
        'Mejores_params': str(best_params_clean)
    }
    grids[nombre] = grid

    print(f'   ✅ Listo en {elapsed/60:.2f} min')
    print(f'   AUC: {auc:.4f} | Accuracy: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}')
    print(f'   Mejores hiperparámetros: {best_params_clean}')

tiempo_total = (time.time() - tiempo_total_inicio) / 60
print(f'\n⏱️  Tiempo total de entrenamiento: {tiempo_total:.2f} min')

> **Análisis:** Describir el rendimiento de cada modelo, qué hiperparámetros resultaron óptimos, cuánto tardó cada uno y si hay diferencias significativas entre ellos.

## 5. Ranking comparativo

In [ ]:
ranking = pd.DataFrame(resultados).T.sort_values('AUC', ascending=False)
display(ranking)

In [ ]:
metricas_plot = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
x      = np.arange(len(ranking))
width  = 0.15
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'goldenrod']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (metrica, color) in enumerate(zip(metricas_plot, colors)):
    ax.bar(x + i * width, ranking[metrica].astype(float), width,
           label=metrica, color=color, edgecolor='black')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(ranking.index, rotation=0)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparación de modelos — todas las métricas')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ranking['AUC'].astype(float).plot(
    kind='bar', figsize=(8, 4), color='steelblue', edgecolor='black'
)
plt.title('Comparación de modelos por AUC')
plt.ylabel('AUC')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
ranking['Tiempo (min)'].astype(float).sort_values().plot(
    kind='barh', figsize=(8, 4), color='coral', edgecolor='black'
)
plt.title('Tiempo de entrenamiento por modelo (minutos)')
plt.xlabel('Minutos')
plt.tight_layout()
plt.show()

> **Análisis:** Comparar los modelos en todas las métricas. Justificar cuál es la métrica principal para este problema de churn (AUC-ROC vs F1 vs Recall) y por qué. Comentar la relación entre tiempo de entrenamiento y ganancia en AUC.

## 6. Evaluación de modelos

### 6.1 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, len(grids), figsize=(22, 4))

for i, (nombre, grid) in enumerate(grids.items()):
    y_pred = grid.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot(
        ax=axes[i], colorbar=False, cmap='Blues'
    )
    axes[i].set_title(nombre)

plt.suptitle('Matrices de confusión', fontsize=13)
plt.tight_layout()
plt.show()

> **Análisis:** Comparar los falsos negativos (predice No Churn cuando es Churn) de cada modelo. En churn, los falsos negativos son los más costosos: el cliente se va sin que la empresa pueda retenerlo.

### 6.2 Classification report

In [ ]:
for nombre, grid in grids.items():
    print(f'\n{"="*50}')
    print(f'  {nombre}')
    print(f'{"="*50}')
    print(classification_report(
        y_test, grid.predict(X_test),
        target_names=['No Churn', 'Churn']
    ))

> **Análisis:** Describir precision, recall y F1 por clase para cada modelo. ¿Cuál tiene mejor recall en la clase Churn?

### 6.3 Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for nombre, grid in grids.items():
    RocCurveDisplay.from_estimator(grid, X_test, y_test, ax=ax, name=nombre)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_title('Curva ROC — Comparación de modelos')
plt.tight_layout()
plt.show()

> **Análisis:** Describir qué modelos tienen mayor área bajo la curva ROC y cómo se separan del clasificador aleatorio.

### 6.4 Importancia de características

Se extraen las 10 variables más importantes de cada modelo para entender qué factores impulsan el churn.

In [ ]:
def get_feature_names(grid):
    """Extrae nombres de features tras OneHotEncoding desde el Pipeline."""
    prep      = grid.best_estimator_.named_steps['prep']
    cat_names = prep.named_transformers_['cat'].get_feature_names_out(categoricas).tolist()
    return cat_names + numericas


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (nombre, grid) in enumerate(grids.items()):
    clf = grid.best_estimator_.named_steps['clf']

    if hasattr(clf, 'feature_importances_'):
        feat_names   = get_feature_names(grid)
        importancias = pd.Series(clf.feature_importances_, index=feat_names)
        top10        = importancias.nlargest(10).sort_values()

        top10.plot(kind='barh', ax=axes[i], color='steelblue', edgecolor='black')
        axes[i].set_title(f'Top 10 features — {nombre}')
        axes[i].set_xlabel('Importancia')
    else:
        axes[i].text(0.5, 0.5, f'{nombre}\nno tiene feature_importances_',
                     ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(nombre)

plt.suptitle('Importancia de características por modelo (Top 10)', fontsize=13)
plt.tight_layout()
plt.show()

> **Análisis:** Identificar qué variables aparecen consistentemente entre las más importantes en los cuatro modelos. ¿Coincide con lo observado en el EDA?

## 7. Guardado del modelo final y archivos para LIME

### 7.1 Selección y guardado del modelo

In [ ]:
mejor_nombre = ranking['AUC'].astype(float).idxmax()
mejor_grid   = grids[mejor_nombre]

print(f'Modelo seleccionado: {mejor_nombre}')
print(f'AUC en test:         {resultados[mejor_nombre]["AUC"]}')
print(f'Mejores hiperparámetros: {resultados[mejor_nombre]["Mejores_params"]}')
print(f'\nPipeline a guardar:')
print(mejor_grid.best_estimator_)

In [ ]:
ruta_modelo = os.path.join('..', 'app', 'model.joblib')
os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)

joblib.dump(mejor_grid.best_estimator_, ruta_modelo)
print(f'✅ Modelo guardado en "{ruta_modelo}"')

### 7.2 CSV de resultados con métricas e hiperparámetros

In [ ]:
os.makedirs('../data', exist_ok=True)

resultados_df = pd.DataFrame(resultados).T.sort_values('AUC', ascending=False)
resultados_df.index.name = 'Modelo'
resultados_df.to_csv('../data/resultados_modelos.csv')

print('✅ Resultados guardados en data/resultados_modelos.csv')
display(resultados_df)

### 7.3 Datos de train/test

In [ ]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv',   index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv',   index=False)

print('✅ Archivos train/test guardados:')
print(f'   - data/X_train.csv ({X_train.shape[0]} filas, {X_train.shape[1]} columnas)')
print(f'   - data/X_test.csv  ({X_test.shape[0]} filas, {X_test.shape[1]} columnas)')

### 7.4 Archivos de metadatos categóricos para LIME

`LimeTabularExplainer` necesita saber cuáles columnas son categóricas (por índice en X original, antes del OHE) y cuáles son sus valores posibles, para generar perturbaciones correctas y mostrar etiquetas legibles en las explicaciones.

In [ ]:
feature_names       = X_train.columns.tolist()
categorical_indices = [feature_names.index(c) for c in categoricas]

categorical_names = {
    feature_names.index(col): sorted(X_train[col].dropna().unique().tolist())
    for col in categoricas
}
categorical_names_str = {str(k): v for k, v in categorical_names.items()}

with open('../data/feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

with open('../data/categorical_features.json', 'w') as f:
    json.dump(categorical_indices, f, indent=2)

with open('../data/categorical_names.json', 'w') as f:
    json.dump(categorical_names_str, f, indent=2)

print('✅ Archivos para LIME guardados:')
print(f'   - data/feature_names.json        ({len(feature_names)} features)')
print(f'   - data/categorical_features.json ({len(categorical_indices)} categóricas)')
print(f'   - data/categorical_names.json')
print(f'\nÍndices categóricos: {categorical_indices}')
print(f'Features: {feature_names}')

## Conclusiones del modelado

> Completar con los hallazgos principales:
> - Modelo con mejor desempeño y justificación de la elección
> - Comparación entre modelos (diferencias en AUC, F1, Recall) y relación con el tiempo de entrenamiento
> - Variables más importantes para predecir el churn
> - Observaciones sobre falsos negativos vs falsos positivos en el contexto del negocio
> - Posibles mejoras: SMOTE para desbalance, umbral de clasificación ajustado, más hiperparámetros